In [1]:
import pandas as pd     
from datetime import datetime


#### Update Path Here


In [2]:
# To get Nielsen, OMT-WG & OMT-F data (MY)
path1 = r"C:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - June\loreal-report-automation (2)\Generated Data\Brand Ranking\JUNE 2026\MY CPD Brand Ranking JUNE 2026.xlsx"

# To get offline estimation data & Top Brands (MY & SG) [CPD only]
path2 = r"C:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - June\loreal-report-automation (2)\Brand Ranking\Raw\Offline Estimation Jun'26_updated with Top NIQP Brands Q2'26.xlsx"     

# To get Nielsen, OMT-WG & OMT-F data (SG)
path3 = r"C:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - June\loreal-report-automation (2)\Generated Data\Brand Ranking\JUNE 2026\SG CPD Brand Ranking JUNE 2026.xlsx"

# For Singapore Tiktok Estimation
path4 = r"C:\Users\balatarsini_avinitya\Downloads\MYSG ONE CPD CMI YTD Jun'26 (2).xlsx"

# To get offline bodycare data (MY)
path5 = r"C:\Users\balatarsini_avinitya\Downloads\Final folders\Offline Bodycare ingestion V 23.06.26\Offline Bodycare ingestion\CPD\CPD Data Output Jul'26.xlsx"

In [3]:
# To determine whether we have actual data for Offline Estimation
actual_data = False

# Manual override: reporting is delayed; 
# e.g.: although the current month is January, only data up to November is included
# Update required here ↓
working_month = 7 #T-1
# curr_year = datetime.now().year - 1         # e.g., 2025
# prev_year = curr_year - 1                   # e.g., 2024
curr_year = 2026         # e.g., 2025
prev_year = 2025
last_year = 2024

# Auto calculation: no manual update required (still required testing)
# working_month = datetime.now().month 
# curr_year = datetime.now().year
# prev_year = datetime.now().year - 1

# if working_month == 1:
#     curr_year -= 1
#     prev_year -= 1
#     working_month = 13

if working_month in [1, 4, 7, 10]:
    actual_data = True


## Helper Functions

In [4]:
# Helper function to reformat Year values
def normalize_time_period(df: pd.DataFrame):
    df['Time Period'] = df['Time Period'].astype(str).str.replace(r'\.0$', '', regex=True)

    # Fix YTD labels: remove any space after 'YTD'
    is_ytd = df['Time Period'].str.contains('YTD')
    df.loc[is_ytd, 'Time Period'] = df.loc[is_ytd, 'Time Period'].str.replace('YTD ', 'YTD', regex=False)
    
    # Add FY prefix to 4-digit numbers
    is_fy = df['Time Period'].str.isdigit() & (df['Time Period'].str.len() == 4)
    df.loc[is_fy, 'Time Period'] = 'FY' + df.loc[is_fy, 'Time Period']
    
    return df

def normalize_brand_names(df: pd.DataFrame):
    if 'Brand' in df.columns:
        df['Brand'] = df['Brand'].apply(
            lambda value: 'TIMEPHORIA' if str(value).strip().upper() == 'TIME PHORIA' else value
        )
    return df

def clean_year(df: pd.DataFrame):
    return normalize_time_period(df)

def validate_time_periods(frames: dict):
    for sheet_name, df in frames.items():
        bad = df['Time Period'].astype(str).str.fullmatch(r'20\d{2}\.0')
        if bad.any():
            examples = sorted(df.loc[bad, 'Time Period'].astype(str).unique())
            raise ValueError(f"Bad Time Period values in {sheet_name}: {examples}. Restart kernel and run all cells from the top.")


In [5]:
# Helper function to add necessary columns
def add_columns(df: pd.DataFrame, flag: bool, online: bool):
    if online:
        df['Channel'] = 'Online'
    else:
        df['Channel'] = 'Offline'
        
    df['Division'] = 'CPD'

    # Add Subdivision column to those dataframes without it
    if flag:
        df['Subdivision'] = 'Exc. Mass Medic'
        
    df = normalize_time_period(df)
    df = normalize_brand_names(df)
    
    # Reordering the columns 
    ordered_df = df[['Category', 'Brand', 'Division', 'Subdivision', 'Time Period', 'SO', 'Channel']]
    
    return ordered_df


In [6]:
# Helper function for market data processing
def clean_market(df1: pd.DataFrame, df2: pd.DataFrame, country: str):
    complete_df = pd.concat([df1, df2])

    if country == 'MY':
        complete_df['Country (Currency)'] = "MY (MYR'000)"
        complete_df['MY/SG'] = 'MY'
    else:
        complete_df['Country (Currency)'] = "SG (SGD'000)"
        complete_df['MY/SG'] = 'SG'

    complete_df = normalize_time_period(complete_df)
    complete_df = normalize_brand_names(complete_df)

    complete_df = complete_df[[
        'Country (Currency)', 
        'MY/SG', 
        'Category', 
        'Brand', 
        'Division', 
        'Subdivision', 
        'Time Period', 
        'SO', 
        'Channel', 
        'Is_Loreal', 
        'Flag'
    ]]

    return complete_df
    

# **Offline**


Part 1
Data Source: Loreal automation → Generated Data → Brand Ranking  
- MY: Included Female & Male Skincare and Makeup  
- SG: Included Female & Male Skincare

In [7]:
def off_pt1(path, sheetname):
    df = pd.read_excel(path, sheet_name=sheetname, header=0)

    # Melt/unpivot: Convert from wide to long format
    df_long = pd.melt(
        df,
        id_vars=['Axes', 'Brand', 'Mass/Mass medic'],  # Columns to keep
        var_name='Time Period',     
        value_name='SO'       
    )

    # Remove rows with FY2021 in Time Period column
    df_long = df_long[df_long['Time Period'] != 'FY2021']

    # Renaming columns
    df_long = df_long.rename(columns={'Mass/Mass medic': 'Subdivision','Axes': 'Category'})

    # Reformat Year: 2023 → FY2023; YTD 2023 → YTD2023
    df_long = clean_year(df_long)
    
    # Keep only rows where Time Period contains these years
    df_long = df_long[df_long['Time Period'].str.contains(f'{curr_year}|{prev_year}|{last_year}')] 

    # Replace specific values
    df_long['Category'] = df_long['Category'].replace({
        'Cosmetics': 'Makeup',
    })

    df_long['Brand'] = df_long['Brand'].replace({
        'TOTAL COSMETIC': 'TOTAL MAKEUP',
        'TOTAL MEN': 'TOTAL MALE SKINCARE',
        'TOTAL WOMEN': 'TOTAL FEMALE SKINCARE',
        'MEDIC MARKET': 'TOTAL FEMALE SKINCARE',
        'LOREAL PARIS': "L'OREAL PARIS",
        "LOREAL DERMO EXPERTISE": "L'OREAL PARIS"       
    })

    df_long['Subdivision'] = df_long['Subdivision'].replace({
        'Non-Mass Medic': 'Exc. Mass Medic',
        'Mass': 'Exc. Mass Medic'
    })

    df_long = add_columns(df_long, False, False)
    
    return df_long


Part 2
Data Source: Offline Est (L'oreal Brands and Market Numbers)
- MY: Included Hair Care, Hair Colour, Suncare
- SG: Included Makeup, Hair Care, Hair Colour, Suncare

In [8]:
def off_pt2(path, sheetname):
    df = pd.read_excel(path, sheet_name=sheetname, header=0)
    
    # Keep necessary columns only 
    if actual_data:
        df = df.filter([
            'Year', 'Brand_x', 'Category_x', 'To be replaced on the Value column',
            'Year.1', 'Brand_x.1', 'Category_x.1', 'To be replaced on the Value column.1',
            'Year.2', 'Brand_x.2', 'Category_x.2', 'To be replaced on the Value column.2'])
    else:
        df = df.filter([
            'Year', 'Brand_x', 'Category_x', 'To be replaced on the Value column',
            'Year.1', 'Brand_x.1', 'Category_x.1', 'Value.1',
            'Year.2', 'Brand_x.2', 'Category_x.2', 'Value.2'])
        
    cols_per_year = 4
        
    py_data = df.iloc[:, 0:cols_per_year].copy()                       # prev prev year
    ly_data = df.iloc[:, cols_per_year:cols_per_year*2].copy()         # prev year
    ty_data = df.iloc[:, cols_per_year*2:cols_per_year*3].copy() # current year
    
    # Add this here
    for part in [py_data, ly_data, ty_data]:
        part.columns = ['Time Period', 'Brand', 'Category', 'SO']
    
    py_data = py_data.rename(columns={
        'Year': 'Time Period',
        'Brand_x': 'Brand',
        'Category_x': 'Category',
        'To be replaced on the Value column': 'SO'
    })
    
    # Renaming columns
    ly_data = ly_data.rename(columns={
        'Year'                              : 'Time Period',
        'Brand_x'                           : 'Brand',
        'Category_x'                        : 'Category',
        'To be replaced on the Value column': 'SO'
    })

    ty_data = ty_data.rename(columns={
        'Year.1'                              : 'Time Period',
        'Brand_x.1'                           : 'Brand',
        'Category_x.1'                        : 'Category'
    })

    if actual_data:
        ty_data = ty_data.rename(columns={'To be replaced on the Value column.1': 'SO'})
    else:
        ty_data = ty_data.rename(columns={'Value.1': 'SO'})

    # Reformat Year: 2023 → FY2023; YTD 2023 → YTD2023
    py_data = clean_year(py_data)
    ly_data = clean_year(ly_data)
    ty_data = clean_year(ty_data)

    # Keep only rows where Time Period contains these years
    py_data = py_data[py_data['Time Period'].str.contains(f'{last_year}')]
    ly_data = ly_data[ly_data['Time Period'].str.contains(f'{curr_year}|{prev_year}')] 
    ty_data = ty_data[ty_data['Time Period'].str.contains(f'{curr_year}|{prev_year}')] 

    # Getting YTD values
    # Getting YTD values
    py_duplicate = py_data.copy()
    ly_duplicate = ly_data.copy()
    ty_duplicate = ty_data.copy()

    # Convert FY → YTD
    py_duplicate['Time Period'] = py_duplicate['Time Period'].str.replace('FY', 'YTD')
    ly_duplicate['Time Period'] = ly_duplicate['Time Period'].str.replace('FY', 'YTD')
    ty_duplicate['Time Period'] = ty_duplicate['Time Period'].str.replace('FY', 'YTD')

    # Only keep PY YTD if LY exists
    py_duplicate['SO ny'] = ly_data['SO']
    py_duplicate = py_duplicate[
        py_duplicate['SO ny'].notna() & (py_duplicate['SO ny'] != 0)
    ].drop(columns=['SO ny'])

    # Only keep LY YTD if TY exists
    ly_duplicate['SO ny'] = ty_data['SO']
    ly_duplicate = ly_duplicate[
        ly_duplicate['SO ny'].notna() & (ly_duplicate['SO ny'] != 0)
    ].drop(columns=['SO ny'])

    # Combine all data
    py_final = pd.concat([py_data, py_duplicate], ignore_index=True)
    ly_final = pd.concat([ly_data, ly_duplicate], ignore_index=True)
    ty_final = pd.concat([ty_data, ty_duplicate], ignore_index=True)

    ay_final = pd.concat([py_final, ly_final, ty_final], ignore_index=True)     # 2024 + 2025 data
    ay_final = add_columns(ay_final, True, False)

    ay_final['Brand'] = ay_final['Brand'].replace({'LOREAL PARIS': "L'OREAL PARIS"})               

    return ay_final


Part 3
Data Source: Offline Est (~L'oreal Brands)
- MY: Included Hair Care, Hair Colour, Suncare
- SG: Included Makeup, Hair Care, Hair Colour, Suncare

In [9]:
def off_pt3(path, sheetname):
    df = pd.read_excel(path, sheet_name=sheetname, header=0)
    
    # Melt/unpivot: Convert from wide to long format
    df_long = pd.melt(
        df,
        id_vars=['Category', 'Brand'], 
        var_name='Time Period',     
        value_name='SO'       
    )

    df_long['Category'] = df_long['Category'].replace({
        'Hair Color': 'Hair Colour'
    })

    df_long = add_columns(df_long, True, False)
    
    return df_long


In [10]:
def off_bodycare(path, sheetname):
    df = pd.read_excel(path, sheet_name=sheetname, header=0)
    df = df.rename(columns={
        'Brand_x': 'Brand',
        'Category_x': 'Category',
        'Value': 'SO'
    })

    if 'MY/SG' in df.columns:
        df = df[df['MY/SG'] == sheetname]

    df = df[df['Year'].isin([last_year, prev_year, curr_year])]

    fy_data = (
        df.groupby(['Category', 'Brand', 'Year'], as_index=False, dropna=False)['SO']
        .sum()
    )
    fy_data['Time Period'] = 'FY' + fy_data['Year'].astype(str)

    ytd_cutoff_month = working_month - 1
    ytd_data = (
        df[df['Month'] <= ytd_cutoff_month]
        .groupby(['Category', 'Brand', 'Year'], as_index=False, dropna=False)['SO']
        .sum()
    )
    ytd_data['Time Period'] = 'YTD' + ytd_data['Year'].astype(str)

    bodycare = pd.concat([fy_data, ytd_data], ignore_index=True)
    bodycare = bodycare[['Category', 'Brand', 'Time Period', 'SO']]
    bodycare['Brand'] = bodycare['Brand'].replace({'MARKET': 'Market'})

    return add_columns(bodycare, True, False)


## **Online-wg/ Online-f**

Data Source: Loreal automation → Generated Data → Brand Ranking  

In [11]:
def online_data(path, sheetname):
    df = pd.read_excel(path, sheet_name=sheetname, header=0)
    df['Subdivision'] = df['Subdivision'].fillna('Exc. Mass Medic')
    df['Subdivision'] = df['Subdivision'].replace({
        'NA': 'Exc. Mass Medic',
        'nan': 'Exc. Mass Medic',
        'Non-Mass Medical': 'Exc. Mass Medic',
        'Mass Medical': 'Mass Medic'
    })
    # Remove unwanted columns 
    df = df.drop(['Universe'], axis=1)

    # To ensure consistent data types
    df = df.astype({
        df.columns[0]: str,     
        df.columns[2]: int,   
        df.columns[3]: str,
        df.columns[4]: str,
        df.columns[5]: str
    })

    # Renaming columns
    df = df.rename(columns={'Year': 'Time Period', 'Total Est. Sales Local': 'SO'})

    # Perform Mapping
    df['Category'] = df['Category L1']
    l2 = df['Category L2'].str.lower().fillna('')

    l2_map = {
    'sun care'  : 'Suncare',
    'makeup'    : 'Makeup', 
    'hair color': 'Hair Colour',
    }

    for keyword, final_cat in l2_map.items():
        mask = l2.str.contains(keyword)
        df.loc[mask, 'Category'] = final_cat

    # Replace specific values
    df['Subdivision'] = df['Subdivision'].replace({
        'nan': 'Exc. Mass Medic',
        'Non-Mass Medical': 'Exc. Mass Medic',
        'Mass Medical'    : 'Mass Medic'
    })

    df['Category'] = df['Category'].replace({
        'FEMALE SKINCARE': 'Female Skincare',
        'MALE SKINCARE'  : 'Male Skincare', 
        'HAIR'           : 'Hair Care'
    })

    # Reformat Year: 2023 → FY2023; YTD 2023 → YTD2023
    df = clean_year(df)
    df = df[df['Time Period'].str.contains(f'{curr_year}|{prev_year}|{last_year}')] 

    # Getting YTD values
    df_duplicate = df.copy()
    df_duplicate['Time Period'] = df_duplicate['Time Period'].str.replace('FY', 'YTD')

    df_duplicate = df_duplicate[df_duplicate['Month'] < working_month]
    # df_duplicate.head(12)
    df_concat = pd.concat([df, df_duplicate], ignore_index=True)
    df_concat = df_concat.drop(['Month'], axis=1)

    df_concat = add_columns(df_concat, False, True)

    return df_concat


Data Source: MYSG ONE CPD CMI YTD for Singapore Tiktok Estimation 

In [12]:
def on_pt2(path):
    df = pd.read_excel(path, sheet_name='O+O Data', header=2)
    
    # Keep necessary columns
    df = df.filter(['Country (Currency)', 'Brand', 'Platform', 'Year', 'Month', 'Brand_x', 'Category_x', 'Value.1'])

    df = df.rename(columns={
        'Brand'     : 'Subdivision',
        'Year'      : 'Time Period',
        'Brand_x'   : 'Brand',
        'Category_x': 'Category',
        'Value.1'   : 'SO'
    })

    # Filter necessary rows
    df = df[df['Country (Currency)'] == "SG (SGD'000)"]
    df = df[df['Platform'] == 'Tiktok']
    df = df[df['Brand'] == 'Market']

    # Replace specific values
    df['Subdivision'] = df['Subdivision'].replace({
        'EXC. MASS MEDIC': 'Exc. Mass Medic',
        'PLUS MASS MEDIC': 'Mass Medic'
    })

    # To ensure consistent data types
    df = df.astype({df.columns[3]: int})

    df = clean_year(df)
    # df = df[df['Time Period'].str.contains(f'{curr_year}|{prev_year}')]
    # df_duplicate = df.copy()

    ly = df[df['Time Period'].str.contains(f'{prev_year}')]
    ty = df[df['Time Period'].str.contains(f'{curr_year}')]
    py = df[df['Time Period'].str.contains(f'{last_year}')]
    ly_copy = ly[ly['Time Period'].str.contains(f'{prev_year}')].copy()
    ty_copy = ty[ty['Time Period'].str.contains(f'{curr_year}')].copy()
    py_copy = py[py['Time Period'].str.contains(f'{last_year}')].copy()
    ly_copy = ly_copy[ly_copy['Month'] < working_month]
    ty_copy = ty_copy[ty_copy['Month'] < working_month]
    py_copy
    ay = pd.concat([ly_copy, ty_copy,py_copy], ignore_index=True)
    ay['Time Period'] = ay['Time Period'].str.replace('FY', 'YTD')

    df_final = pd.concat([df, ay], ignore_index=True)
    df_final = df_final.drop(['Country (Currency)', 'Platform', 'Month'], axis=1)

    df_final = add_columns(df_final, False, True)

    return df_final


## **O+O-wg/ O+O-f**

Data Source: Offline + Online

In [13]:
def oo_data(offline, online_type, flag_type, rmv_year):
    df_concat = pd.concat([offline, online_type], ignore_index=True)

    # Replace specific values
    df_concat['Division'] = df_concat['Division'].replace({
        'ACD': 'LDB'
    })

    df_concat['Category'] = df_concat['Category'].replace({
        'MAKEUP': 'Makeup'
    })

    df_concat['Subdivision'] = df_concat['Subdivision'].replace({
        'Exc. Mass medic': 'Exc. Mass Medic',
        'Mass medic'     : 'Mass Medic'
    })

    # # Filter out rows with FY2024 values for MY only
    # if rmv_year:
    #     df_concat = df_concat[df_concat['Time Period'] != 'FY2024']

    # Specifying L'oreal Brands
    loreal_brands = [
        "L'Oreal Paris",
        "Maybelline",
        "Garnier",
        "La Roche Posay"
        "Vichy",
        "Skin Ceuticals",
        "L'OREAL HAIR EXPERTISE",
        "ORIGINAL",
        "BOTANICAL FRESH CARE",
        "LOREAL PARIS",
        "3CE"
    ]

    # Convert to lowercase for case-insensitive matching
    loreal_brands_lower = [brand.lower() for brand in loreal_brands]

    # To check for L'oreal Brands
    df_concat["Is_Loreal"] = df_concat["Brand"].apply(
        lambda x: "Yes" if str(x).lower() in loreal_brands_lower else "No"
    )

    # Add flag column with flag_type values 
    df_concat['Flag'] = flag_type

    # For Garnier (WG) only - Makeup → Female Skincare; Hair Care → Hair Colour 
    if 'WG' in df_concat['Flag'].values:
        garnier_mask = df_concat['Brand'].str.contains('Garnier', case=False, na=False)
        df_concat.loc[garnier_mask & (df_concat['Category'] == 'Makeup'), 'Category'] = 'Female Skincare'
        df_concat.loc[garnier_mask & (df_concat['Category'] == 'Hair Care'), 'Category'] = 'Hair Colour'
    
    # Remove Market numbers for WG only
    if flag_type == 'WG':
        df_concat = df_concat[df_concat['Brand'] != 'Market']

    return df_concat


To find Market Data 

In [14]:
def total_market(df: pd.DataFrame, word: str|None, offline: bool):
    # For offline
    if offline:
        if word == 'Market':
            market_df = df[(df['Brand'] == 'Market') & (df['Channel'] == 'Offline') & (df['Flag'] == 'F')].copy()
        else:
            market_df = df[(df['Category'] == word) & (df['Channel'] == 'Offline') & (df['Flag'] == 'F')].copy()
            market_df['Brand'] = 'Market'
            
    # For online
    else:
        market_df = df[(df['Channel'] == 'Online') & (df['Flag'] == 'F')].copy()
        market_df['Brand'] = 'Market'

    group_columns = ['Category', 'Brand', 'Division', 'Subdivision', 'Time Period', 'Channel', 'Flag']
    agg_df = market_df.groupby(group_columns, as_index=False, dropna=False).agg({'SO': 'sum'})
    agg_df['Is_Loreal'] = 'No'

    return agg_df


**MY FINAL DATA [CPD]**


In [15]:
my_offline1 = off_pt1(path1, 'Nielsen')
my_offline2 = off_pt2(path2, 'MY')
my_offline3 = off_pt3(path2, 'MY Top Brands')
my_offline4 = off_bodycare(path5, 'MY')

# Getting complete offline data
my_offline  = pd.concat([my_offline1, my_offline2],  ignore_index=True)
my_final_off = pd.concat([my_offline, my_offline3, my_offline4], ignore_index=True)

# Getting online data
my_on_wg = online_data(path1, 'OMT-WG')
my_on_f  = online_data(path1, 'OMT-F' )

# Getting complete O+O data
my_oo_wg = oo_data(my_final_off, my_on_wg, 'WG', True)
my_oo_f  = oo_data(my_final_off, my_on_f, "F", True)

# Combine oo_f under oo_wg to get final oo_wg
my_oo_wg_final = pd.concat([my_oo_wg, my_oo_f], ignore_index=True)


In [16]:
# Save output
my_outputs = {
    'Offline': my_final_off,
    'Online-wg': my_on_wg,
    'Online-f': my_on_f,
    'O+O-wg': my_oo_wg_final,
    'O+O-f': my_oo_f,
}
validate_time_periods(my_outputs)

with pd.ExcelWriter('MY CPD Brand Ranking.xlsx', engine='openpyxl') as writer:
    for sheet_name, df in my_outputs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)


**SG FINAL DATA [CPD]**

In [17]:
sg_offline1 = off_pt1(path3, 'Nielsen')
sg_offline2 = off_pt2(path2, 'SG')
sg_offline3 = off_pt3(path2, 'SG Top Brands')

# Getting complete offline data
sg_offline = pd.concat([sg_offline1, sg_offline2],  ignore_index=True)
sg_final_off = pd.concat([sg_offline, sg_offline3], ignore_index=True)

# Getting online data
sg_on_wg = online_data(path3, 'OMT-WG')
sg_on_f  = online_data(path3, 'OMT-F' )
on_sg = on_pt2(path4)
sg_on_f = pd.concat([on_sg, sg_on_f], ignore_index=True)

# Getting complete O+O data
sg_oo_wg = oo_data(sg_final_off, sg_on_wg, 'WG', False)
sg_oo_f  = oo_data(sg_final_off, sg_on_f, "F", False)

# Combine oo_f under oo_wg to get final oo_wg
sg_oo_wg_final = pd.concat([sg_oo_wg, sg_oo_f], ignore_index=True)


In [18]:
# Save output
sg_outputs = {
    'Offline': sg_final_off,
    'Online-wg': sg_on_wg,
    'Online-f': sg_on_f,
    'O+O-wg': sg_oo_wg_final,
    'O+O-f': sg_oo_f,
}
validate_time_periods(sg_outputs)

with pd.ExcelWriter('SG CPD Brand Ranking.xlsx', engine='openpyxl') as writer:
    for sheet_name, df in sg_outputs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    

**Split Data MYSG CPD Combined**

In [19]:
# To get offline market data (MY)
my_female = total_market(my_oo_f, 'Female Skincare', True)
my_male = total_market(my_oo_f, 'Male Skincare', True)
my_makeup = total_market(my_oo_f, 'Makeup', True)
my_market_numbers = total_market(my_oo_f, 'Market', True)

my_data1 = pd.concat([my_female, my_male], ignore_index=True)
my_data2 = pd.concat([my_makeup, my_data1], ignore_index=True)
my_data3 = pd.concat([my_data2, my_market_numbers], ignore_index=True)
my_final_offline = pd.concat([my_data3, my_oo_wg], ignore_index=True)

# To get online market data
my_final_online = total_market(my_oo_f, None, False)

# To get online + offline data
my_final_oo = clean_market(my_final_offline, my_final_online, 'MY')


In [20]:
# To get offline market data (SG)
sg_female = total_market(sg_oo_f, 'Female Skincare', True)
sg_male = total_market(sg_oo_f, 'Male Skincare', True)
sg_market_numbers = total_market(sg_oo_f, 'Market', True)

sg_data1 = pd.concat([sg_female, sg_male], ignore_index=True)
sg_data2 = pd.concat([sg_data1, sg_market_numbers], ignore_index=True)
sg_final_offline = pd.concat([sg_data2, sg_oo_wg], ignore_index=True)

# To get online market data
sg_final_online = total_market(sg_oo_f, None, False)

# To get online + offline data
sg_final_oo = clean_market(sg_final_offline, sg_final_online, 'SG')

# To get MYSG O+O Data
final_oo = pd.concat([my_final_oo, sg_final_oo])


In [21]:
combined_outputs = {
    'MY CPD': my_final_oo,
    'SG CPD': sg_final_oo,
    'MY & SG CPD': final_oo,
}
validate_time_periods(combined_outputs)

with pd.ExcelWriter('Split Data MYSG CPD Combined.xlsx', engine='openpyxl') as writer:
    for sheet_name, df in combined_outputs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    